# Mini-projet Séries Temporelles — Partie 5 : réseaux récurrents LSTM et GRU (section 4.1)
Prérequis : `01_eda.ipynb` (jeu de données), `02_sarima.ipynb` (prévisions de référence dans `results/forecasts.csv`). Code en **Python seulement** (TensorFlow/Keras) : l'énoncé autorise un seul langage pour le socle avancé.

**Choix de conception (fixés avant de regarder les résultats).**
1. **Cible = variation annuelle** `z_t = log(core_t) − log(core_{t−12})`, pas le niveau. Un réseau n'extrapole pas une tendance ; en prédisant la variation annuelle, on part du niveau et de la saisonnalité de l'an dernier et le réseau n'apprend que l'écart de croissance. Le niveau est reconstruit : `log(core_t) = log(core_{t−12}) + z_t`.
2. **Entrées (NARX)** à chaque pas : la cible retardée `z_{t−1}`, sin/cos du mois, l'indicatrice COVID annualisée, et — pour les modèles « avec chaleur » — la variation annuelle des degrés-jours `dcdd_t = cdd24_t − cdd24_{t−12}`. Les modèles « univariés » n'ont pas la chaleur : ils servent d'ablation.
3. **Chaleur future, deux scénarios** (comme pour le SARIMAX) : *climatologie* (réalisable : aucune météo future) et *oracle* (météo réelle du test : prévision conditionnelle, non opérationnelle).
4. **Prévision récursive** à 24 pas depuis fin août 2022, mêmes métriques que le reste du projet.
5. **Validation chronologique** : les dernières fenêtres de l'apprentissage servent de validation (jamais de k-fold), puis validation croisée **walk-forward** à origine glissante (section 7).
6. **Références** : naïf saisonnier, SARIMA, et une **régression linéaire avec exactement les mêmes entrées** — pour mesurer ce que la non-linéarité du réseau apporte réellement.

In [ ]:
import os, time, warnings, platform, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
from common import load, split, metrics, save_forecasts
from dl_utils import (FEATS, static_features, cdd_climatology, training_frame, fit_nn, forecast_recursive,
                      nn_predictor, fit_ols, ols_predictor)
warnings.filterwarnings("ignore")
SEED = 42; random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
print("python", platform.python_version(), "| tensorflow", tf.__version__, "| statsmodels", statsmodels.__version__, "| CPU only")

df = load(); df["ly"] = np.log(df["core"]); train, test = split(df); H = len(test); ly = train["ly"]
st_clim = static_features(df.index, cdd_climatology(train, df.index), df["covid_2020_04"])   # chaleur future = climatologie du train
st_orac = static_features(df.index, df["cdd24"], df["covid_2020_04"])                       # chaleur future = observée (oracle)
D = training_frame(ly, st_clim)
print(len(D), "observations d'apprentissage utilisables | corr(z, dcdd) =", round(D["z"].corr(D["dcdd"]), 2),
      "| corr(z, z retardé) =", round(D["z"].corr(D["zlag"]), 2))

## 1. Référence linéaire avec les mêmes entrées

In [ ]:
def rmse_h(pred, h): return metrics(test["core"], pred, h)["RMSE"]
lin = {}
o_u, o_c = fit_ols(ly, st_clim, "univarié"), fit_ols(ly, st_clim, "avec chaleur")
lin["Linéaire univarié"] = forecast_recursive(ols_predictor(o_u), ly, st_clim, o_u["feats"], 1, H)
lin["Linéaire + chaleur (clim.)"] = forecast_recursive(ols_predictor(o_c), ly, st_clim, o_c["feats"], 1, H)
lin["Linéaire + chaleur (oracle)"] = forecast_recursive(ols_predictor(o_c), ly, st_orac, o_c["feats"], 1, H)
naif = pd.read_csv("results/forecasts.csv", index_col=0, parse_dates=True)["Naïf saisonnier"]
rows = [{"modèle": "Naïf saisonnier", "RMSE h=10": rmse_h(naif, 10), "RMSE h=24": rmse_h(naif, 24)}] + \
       [{"modèle": k, "RMSE h=10": rmse_h(v, 10), "RMSE h=24": rmse_h(v, 24)} for k, v in lin.items()]
display(pd.DataFrame(rows).set_index("modèle").round(1))
print("coefficients (variables : constante,", o_c["feats"], ") :", np.round(o_c["beta"], 4))

## 2. Choix des hyperparamètres (validation chronologique)
Grille : fenêtre w ∈ {6, 12}, unités ∈ {8, 16}, dropout ∈ {0, 0,2}, pas d'apprentissage ∈ {0,003 ; 0,01}, pour chacune des 4 combinaisons (LSTM/GRU × univarié/avec chaleur). Critère : perte de validation (MSE sur les **24 dernières fenêtres de l'apprentissage**, en unités standardisées), arrêt précoce avec au moins 5 époques. Un seul germe (42) pendant la recherche ; la stabilité est évaluée ensuite sur 5 germes.

In [ ]:
t0 = time.time(); rows = []
for arch in ("LSTM", "GRU"):
    for fs in ("univarié", "avec chaleur"):
        for w in (6, 12):
            for units in (8, 16):
                for dr in (0.0, 0.2):
                    for lr in (3e-3, 1e-2):
                        info = fit_nn(arch, fs, ly, st_clim, w, units, dr, SEED, lr=lr)
                        rows.append(dict(arch=arch, featset=fs, w=w, units=units, dropout=dr, lr=lr,
                                         val_loss=info["val_loss"], val_baseline=info["val_baseline"], epochs=info["best_epoch"]))
grid = pd.DataFrame(rows)
best_cfg = grid.loc[grid.groupby(["arch", "featset"])["val_loss"].idxmin()].set_index(["arch", "featset"])
print(f"{len(grid)} entraînements en {time.time() - t0:.0f} s")
display(best_cfg.round(3))
gb = grid.groupby(["arch", "featset"]).agg(meilleure_val=("val_loss", "min"), prediction_constante=("val_baseline", "first"),
                                              configs_meilleures_que_constante=("val_loss", lambda s: 0))
for k in gb.index: gb.loc[k, "configs_meilleures_que_constante"] = int((grid[(grid.arch == k[0]) & (grid.featset == k[1])].eval("val_loss < val_baseline")).sum())
gb["gain vs constante (%)"] = 100 * (1 - gb["meilleure_val"] / gb["prediction_constante"])
display(gb.round(3))

## 3. Courbes de perte (apprentissage / validation) des configurations retenues

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharey=False)
for a, ((arch, fs), cfg) in zip(ax.ravel(), best_cfg.iterrows()):
    info = fit_nn(arch, fs, ly, st_clim, int(cfg.w), int(cfg.units), cfg.dropout, SEED, lr=cfg.lr)
    a.plot(info["history"]["loss"], label="apprentissage"); a.plot(info["history"]["val_loss"], label="validation")
    a.axvline(info["best_epoch"] - 1, color="k", ls=":", lw=.8); a.set_title(f"{arch} — {fs} (w={int(cfg.w)}, {int(cfg.units)} unités)", fontsize=9)
    a.set_xlabel("époque"); a.set_ylabel("MSE (standardisé)"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Prévisions sur l'échantillon test (ensemble de 5 germes)
Pour chaque configuration retenue : réentraînement sur **toutes** les fenêtres d'apprentissage avec le nombre d'époques sélectionné, 5 germes différents, prévision récursive à 24 pas ; la prévision du modèle est la **moyenne géométrique** des 5 trajectoires, et on garde la dispersion entre germes.

In [ ]:
SEEDS = [42, 43, 44, 45, 46]; t0 = time.time()
F, spread = {}, []
for (arch, fs), cfg in best_cfg.iterrows():
    runs = {"clim": [], "orac": []}
    for sd in SEEDS:
        info = fit_nn(arch, fs, ly, st_clim, int(cfg.w), int(cfg.units), cfg.dropout, sd, epochs=int(cfg.epochs), lr=cfg.lr)
        pr = nn_predictor(info)
        runs["clim"].append(forecast_recursive(pr, ly, st_clim, info["feats"], int(cfg.w), H))
        if fs == "avec chaleur": runs["orac"].append(forecast_recursive(pr, ly, st_orac, info["feats"], int(cfg.w), H))
    scen = [("clim", "(clim.)" if fs == "avec chaleur" else "")] + ([("orac", "(oracle)")] if fs == "avec chaleur" else [])
    for key, lab in scen:
        name = f"{arch} {'+ chaleur' if fs == 'avec chaleur' else 'univarié'} {lab}".strip()
        L = np.log(np.column_stack([r.values for r in runs[key]]))
        F[name] = pd.Series(np.exp(L.mean(axis=1)), index=test.index)
        per = [rmse_h(r, 24) for r in runs[key]]
        spread.append({"modèle": name, "RMSE h=24 par germe : min": min(per), "max": max(per), "écart-type": np.std(per, ddof=1)})
print(f"{len(best_cfg) * len(SEEDS)} entraînements en {time.time() - t0:.0f} s")
allf = {**{"Naïf saisonnier": naif}, **lin, **F}
res = pd.DataFrame({n: {**{f"{k} h=10": v for k, v in metrics(test["core"], p, 10).items()}, **{f"{k} h=24": v for k, v in metrics(test["core"], p, 24).items()}} for n, p in allf.items()}).T
display(res[["RMSE h=10", "MAE h=10", "MAPE h=10", "RMSE h=24", "MAE h=24", "MAPE h=24"]].round(1))
display(pd.DataFrame(spread).set_index("modèle").round(1))

In [ ]:
sarima = pd.read_csv("results/forecasts.csv", index_col=0, parse_dates=True)
fig, ax = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True)
for a, names in zip(ax, [["LSTM univarié", "GRU univarié", "LSTM + chaleur (clim.)", "GRU + chaleur (clim.)"],
                         ["LSTM + chaleur (oracle)", "GRU + chaleur (oracle)"]]):
    a.plot(train.index[-36:], train["core"].iloc[-36:], color="k", lw=1); a.plot(test.index, test["core"], color="k", lw=1.8, label="observé")
    a.plot(test.index, naif, color="gray", ls=":", label="naïf saisonnier"); a.plot(test.index, sarima["SARIMA manuel"], color="tab:blue", ls="--", lw=1, label="SARIMA manuel")
    for n in names: a.plot(test.index, F[n], label=n)
    a.legend(ncol=3, fontsize=8); a.set_ylabel("GWh / mois")
ax[0].set_title("Prévisions à 24 pas — réseaux sans météo future"); ax[1].set_title("Avec la chaleur réelle du test (oracle, non opérationnel)")
plt.tight_layout(); plt.show()

In [ ]:
# sauvegarde pour le tableau comparatif et Diebold-Mariano (section 6)
for n, p in {**lin, **F}.items(): save_forecasts(n, p)
res.round(3).to_csv("results/metrics_dl.csv"); pd.DataFrame(spread).round(3).to_csv("results/spread_dl.csv", index=False)
best_cfg.round(4).to_csv("results/dl_best_config.csv")
print("sauvé : results/forecasts.csv (+ colonnes LSTM/GRU/linéaire), metrics_dl.csv, spread_dl.csv, dl_best_config.csv")

## 5. Validation croisée temporelle walk-forward
7 origines (août 2018, févr. 2019, …, août 2021), horizon 12 mois, **fenêtre d'apprentissage extensible** : à chaque origine on réentraîne tout (réseaux, SARIMA, linéaire) avec les données jusqu'à l'origine seulement et la climatologie recalculée jusqu'à l'origine. Les hyperparamètres sont ceux de la section 2 (sélectionnés avec des données antérieures à août 2022 : un léger biais optimiste est possible pour les réseaux). 3 germes par réseau. **Les mois COVID** (mars-mai 2020, avril 2021) tombent dans les plis des origines févr. 2020 à août 2020 et févr. 2021 ; l'indicatrice COVID est donnée aux réseaux et au SARIMA.

In [ ]:
origins = ["2018-08", "2019-02", "2019-08", "2020-02", "2020-08", "2021-02", "2021-08"]; HF = 12; CV_SEEDS = [42, 43, 44]
cv_rows = []; t0 = time.time()
for og in origins:
    known = df.loc[:og]; n0 = len(known); fut = df.iloc[n0:n0 + HF]; act = fut["core"]
    st = static_features(df.index, cdd_climatology(known, df.index), df["covid_2020_04"]); lyk = known["ly"]
    row = {"origine": og}
    rm = lambda p: float(np.sqrt(np.mean((act.values - np.asarray(p)) ** 2)))
    row["Naïf saisonnier"] = rm(np.exp(lyk.values[-12:]))
    sx = SARIMAX(lyk, exog=known[["covid_2020_04"]], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
    row["SARIMA manuel"] = rm(np.exp(sx.get_forecast(HF, exog=fut[["covid_2020_04"]]).predicted_mean))
    for fs, nm in (("univarié", "Linéaire univarié"), ("avec chaleur", "Linéaire + chaleur")):
        o = fit_ols(lyk, st, fs); row[nm] = rm(forecast_recursive(ols_predictor(o), lyk, st, o["feats"], 1, HF))
    for (arch, fs), cfg in best_cfg.iterrows():
        P = []
        for sd in CV_SEEDS:
            info = fit_nn(arch, fs, lyk, st, int(cfg.w), int(cfg.units), cfg.dropout, sd, epochs=int(cfg.epochs), lr=cfg.lr)
            P.append(np.log(forecast_recursive(nn_predictor(info), lyk, st, info["feats"], int(cfg.w), HF).values))
        row[f"{arch} {'+ chaleur' if fs == 'avec chaleur' else 'univarié'}"] = rm(np.exp(np.mean(P, axis=0)))
    cv_rows.append(row)
cv = pd.DataFrame(cv_rows).set_index("origine"); print(f"validation walk-forward terminée en {time.time() - t0:.0f} s")
display(cv.round(0))
summ = pd.DataFrame({"RMSE moyen": cv.mean(), "écart-type entre plis": cv.std(), "RMSE moyen hors plis COVID": cv.drop(["2020-02", "2020-08", "2021-02"]).mean(),
                     "plis battant le naïf (sur 7)": (cv.lt(cv["Naïf saisonnier"], axis=0)).sum()}).sort_values("RMSE moyen")
display(summ.round(1)); cv.round(1).to_csv("results/cv_walkforward.csv")